# Export PREPARE_GDB by office service province — revised V2

## Purpose and data flow

This is the complete bootstrap exporter, revised from
`04_Export_Prepare_to_Archive(2).ipynb`. Run with the **ArcGIS Pro Python 3**
environment (ArcPy, pandas and openpyxl). PREPARE_GDB is read-only.

1. Inventory `A<BRANCH_CODE>_<ZONE>`; validate polygon schema and WKID.
2. Derive `PROV_ID = BRANCH_CODE[:2]` from the feature-class name.
3. Route the entire feature class to `ARCHIVE_DIR/P<PROV_ID>.gdb`.
4. Preserve each row's administrative `CHANGWAT_CODE`. Validate amphoe/tambon
   against that code, independently of the office's service province.
5. Write temporary C/Z with one file-geodatabase edit session. Record rejected
   OIDs in SQLite and count every outcome by service/administrative province.
6. Reconcile source = C + Z + rejected; publish only nonempty C/Z; checkpoint.

| Source / attributes | Expected result |
|---|---|
| A10000000_47, CHANGWAT_CODE=10, otherwise valid ACTIVE | P10.gdb / C10000000_47; administrative code stays 10 |
| A10000000_47, CHANGWAT_CODE=13, otherwise valid ACTIVE | P10.gdb / C10000000_47; administrative code stays 13 |
| Same cross-province row with a valid non-active status | P10.gdb / Z10000000_47; status and administrative code preserved |
| Missing or malformed CHANGWAT_CODE | Report-only rejection; no branch-based fill |
| Row BRANCH_CODE contradicts feature-class name | Report-only rejection; never silently reroute |

## Classification and fields

- Fatal structural errors are report-only rejections (no C/Z geometry).
- Valid non-active RECORD_STATUS values are preserved in Z.
- Other storable rows with review issues become PENDING_REVIEW in Z.
- Other valid rows become ACTIVE in C.
- A service/administrative province difference is informational and never a
  rejection/review reason by itself.
- CHANGWAT_CODE remains TEXT(2), non-nullable. Validation checks code syntax and
  hierarchy; it does not assert membership in an external province-code catalog.
- PROV_ID is derived metadata, not an added polygon field. The canonical field
  list, APID construction, geometric hashing and deterministic ID rules are retained.
- Existing valid ENTITY_ID/VERSION_ID are preserved. ATTR_HASH includes the
  administrative province, so correcting that value can change generated VERSION_ID.

## How to run

1. Restart the ArcGIS Pro notebook kernel. Review the parameters below.
2. Leave RUN_EXPORT=False and run all cells (or execute the .py script). This
   inventories sources without creating archives, reports, or SQLite state.
3. For the first write, set SOURCE_NAMES=["A10000000_47"] (or another affected
   source), RUN_EXPORT=True, and keep PILOT_FIRST_N=1. Rerun all cells.
4. Inspect the C/Z attributes and ADMIN_PROVINCE_SUMMARY. A valid cross-province
   row must keep CHANGWAT_CODE=13 and go to P10.gdb.
5. Set SOURCE_NAMES=None and PILOT_FIRST_N=None to process all sources. Reuse
   the same V2 state only while the source and export policy remain unchanged.

Default staging output:
`D:\A02-Projects\Refine\03_Archives\_rebuild_service_area_v2\P10.gdb`

Reports and state use a separate `05_reports/2022-05/service_area_v2` directory.
No previous archive or checkpoint is deleted, appended to, or automatically
promoted. Review/reconcile the rebuilt outputs before a separately planned
replacement of old archive datasets. Do not point fresh state at existing C/Z
and expect overwriting; that is deliberately blocked.

## Resume and report interpretation

- The V2 SQLite database stores its schema version, policy fingerprint, source
  root and archive root. Old/foreign state is rejected before checkpoint writes.
- COMPLETED outputs are checked and skipped; BUILT/PUBLISHED outputs resume
  publication; FAILED/BUILDING outputs rebuild only when finals are absent.
- Checkpoint mismatches preserve the original checkpoint and record a separate
  run error. The old zero-output reset switch has been removed.
- Keep PREPARE_GDB immutable during resume. Counts/schema checks cannot detect
  same-count attribute or geometry edits. If it changes, use fresh state and
  a fresh staging directory and rebuild the affected feature classes.
- RUN_SUMMARY and GDB_SUMMARY include completed exports only. FEATURE_CLASSES
  shows all checkpoint states. ADMIN_PROVINCE_SUMMARY groups row outcomes by
  source, PROV_ID and CHANGWAT_CODE, with checkpoint status attached.
- REJECTED_PREVIEW is bounded; `rejected_rows` in SQLite contains all flushed
  rejected OIDs/reasons. Failed runs may contain partial diagnostics; compare
  checkpoint status. Validation rule counts can overlap; do not sum them as rows.
- RUN_ERRORS records errors per attempt, including resume/output mismatches.
- A source whose every row is rejected is FAILED, not a successful empty load.

## Scope and validation limits

This exporter does not spatially assign administrative codes, repair source
geometry, deduplicate across offices, or restore polygons deleted upstream.
Review any earlier branch-province boundary deletion rules separately.
Incremental acquisition merges and relationship creation remain outside this
bootstrap exporter; the empty ENTITY_RELATIONSHIP schema is still initialized.
Do not run two exporters concurrently against these same targets/checkpoints.

Local delivery checks cover syntax, notebook format, transformations with
synthetic geometry, SQLite checkpoints/reporting, and simulated C/Z publication.
Actual ArcPy/geodatabase execution must be verified in your ArcGIS Pro kernel.

ArcPy references:
- [Editor](https://doc.esri.com/en/arcgis-pro/latest/arcpy/data-access/editor.html)
- [Get Count](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/data-management/get-count.html)
- [EnvManager](https://pro.arcgis.com/en/pro-app/3.6/arcpy/classes/envmanager.htm)


## 1. Parameters

In [1]:
from pathlib import Path

# 1. INPUTS AND OUTPUTS
ACQUIRED = "2022-05"
BASE_DIR = Path(r"D:\A02-Projects\Refine")
PREPARE_GDB = BASE_DIR / "01_acquired" / f"A{ACQUIRED}.gdb"
ARCHIVE_ROOT = BASE_DIR / "99_temp"

# Deliberately rebuild separately from existing Pxx.gdb outputs. This script
# does not promote staging outputs or replace the previous archive.
ARCHIVE_DIR = ARCHIVE_ROOT / "_rebuild_service_area_v2"
REPORT_DIR = BASE_DIR / "05_reports" / ACQUIRED / "service_area_v2"
EXPORT_REPORT = REPORT_DIR / f"04_{ACQUIRED}-Export-service-area-v2.xlsx"
Empty_REPORT = EXPORT_REPORT  # Compatibility alias used in the previous notebook.
STATE_DB = REPORT_DIR / f"04_{ACQUIRED}-Export-service-area-v2.state.sqlite"
WORK_GDB = REPORT_DIR / f"04_{ACQUIRED}-Export-service-area-v2_Work.gdb"

# 2. EXECUTION
RUN_EXPORT = True            # First run: inventory only; no output writes.
STOP_ON_ERROR = False
OVERWRITE_EXISTING = False      # Must remain False: no automatic replacement.
PILOT_FIRST_N = None               # None = all selected feature classes.
SOURCE_NAMES = None             # Example: ["A10000000_47"]; None = all sources.

# 3. VALIDATION AND PERFORMANCE
RUN_CHECK_GEOMETRY = True
PROGRESS_EVERY = 100_000
WRITE_REPORT_AFTER_EACH_FEATURE_CLASS = True
REJECTED_PREVIEW_LIMIT = 10_000  # Excel sample; complete rejected OIDs in SQLite.
REJECT_BATCH_SIZE = 10_000
ALLOW_Z_OR_M = False
EXPECTED_WKID = {47: 24047, 48: 24048}
ALLOWED_SCALES = {500, 1000, 2000, 4000, 5000}

# 4. RESUME CONTRACT
# Bump the policy version if transformation semantics are edited later.
# Never edit PREPARE_GDB while resuming. Attribute/geometry edits that retain
# its row count and schema cannot be detected by the lightweight source check.
EXPORT_POLICY_VERSION = "SERVICE_AREA_ADMIN_V2"
STATE_SCHEMA_VERSION = 2

## 2. Imports and canonical schema

In [2]:
import hashlib
import json
import os
import re
import sqlite3
import traceback
import unicodedata
import uuid
from collections import Counter, defaultdict
from contextlib import closing
import datetime as _datetime
from decimal import Decimal, ROUND_HALF_UP

import arcpy
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, "to_string") else value)


FC_PATTERN = re.compile(r"^A(?P<branch>\d{8})_(?P<zone>47|48)$", re.IGNORECASE)
VALID_STATUSES = {"ACTIVE", "SUPERSEDED", "RETIRED", "PENDING_REVIEW", "REJECTED"}
NON_ACTIVE_STATUSES = VALID_STATUSES - {"ACTIVE"}

# Fixed project namespace: deterministic IDs remain stable when an interrupted
# feature class is rebuilt from the same immutable PREPARE_GDB and source OID.
UUID_NAMESPACE = uuid.UUID("d4f612e0-7a9e-5a4e-a993-4c94611fcb80")

# name, ArcGIS field type, length, nullable
CANONICAL_FIELDS = [
    ("CHANGWAT_CODE", "TEXT", 2, False),
    ("AMPHOE_CODE", "TEXT", 4, True),
    ("TAMBON_CODE", "TEXT", 6, True),
    ("LANDOFFICE", "LONG", None, False),
    ("BRANCH_CODE", "TEXT", 8, False),
    ("RAWANG_NAME", "TEXT", 200, True),
    ("INSIDE_X", "DOUBLE", None, False),
    ("INSIDE_Y", "DOUBLE", None, False),
    ("UTM1", "TEXT", 4, True),
    ("UTM2", "LONG", None, True),
    ("UTM3", "TEXT", 4, True),
    ("UTM4", "TEXT", 3, True),
    ("UTMSCALE", "LONG", None, True),
    ("LAND_NO", "TEXT", 20, True),
    ("PARCEL_TYPE", "LONG", None, True),
    ("DOC_TYPE", "LONG", None, True),
    ("LAST_UPD_D", "TEXT", 14, True),
    ("REMARK", "TEXT", 250, True),
    ("APID", "TEXT", 68, True),
    ("ENTITY_ID", "TEXT", 36, True),
    ("VERSION_ID", "TEXT", 36, False),
    ("ACQUIRED", "TEXT", 9, False),
    ("RECORD_STATUS", "TEXT", 20, False),
    ("GEOM_HASH", "TEXT", 64, False),
    ("ATTR_HASH", "TEXT", 64, False),
]

CANONICAL_NAMES = [field[0] for field in CANONICAL_FIELDS]
SOURCE_CANDIDATE_FIELDS = [
    "CHANGWAT_CODE", "AMPHOE_CODE", "TAMBON_CODE", "LANDOFFICE",
    "BRANCH_CODE", "RAWANG_NAME", "UTM1", "UTM2", "UTM3", "UTM4",
    "UTMSCALE", "LAND_NO", "PARCEL_TYPE", "DOC_TYPE", "LAST_UPD_D",
    "REMARK", "ENTITY_ID", "VERSION_ID", "RECORD_STATUS",
]

ATTR_HASH_FIELDS = [
    "CHANGWAT_CODE", "AMPHOE_CODE", "TAMBON_CODE", "LANDOFFICE",
    "BRANCH_CODE", "UTM1", "UTM2", "UTM3", "UTM4", "UTMSCALE",
    "LAND_NO", "PARCEL_TYPE", "DOC_TYPE",
]

RELATIONSHIP_FIELDS = [
    ("RELATIONSHIP_ID", "TEXT", 36, False),
    ("FROM_ENTITY_ID", "TEXT", 36, False),
    ("TO_ENTITY_ID", "TEXT", 36, False),
    ("RELATIONSHIP_TYPE", "TEXT", 20, False),
    ("EFFECTIVE_ACQUIRED", "TEXT", 9, False),
    ("NOTICE", "TEXT", 250, True),
]

def utc_now():
    return _datetime.datetime.now(_datetime.timezone.utc).isoformat(timespec="seconds")

def as_path(value):
    return os.fspath(value)

## 3. Normalization, UTM validation and hashes

In [3]:
def normalize_unicode(value):
    return unicodedata.normalize("NFC", value)


def clean_text(value, max_length=None):
    '''Return (normalized_value, error_code). Never silently truncate.'''
    if value is None:
        return None, None
    text = normalize_unicode(str(value).strip())
    if text == "":
        return None, None
    if max_length is not None and len(text) > max_length:
        return None, f"TEXT_OVERFLOW_{max_length}"
    return text, None


def integer_value(value):
    '''Coerce only exact integers; reject fractional or nonnumeric values.'''
    if value is None or (isinstance(value, str) and value.strip() == ""):
        return None, None
    try:
        number = Decimal(str(value).strip())
    except Exception:
        return None, "NOT_INTEGER"
    integral = number.to_integral_value()
    if number != integral:
        return None, "NOT_INTEGER"
    try:
        return int(integral), None
    except Exception:
        return None, "INTEGER_OVERFLOW"


def digit_text(value, lengths):
    '''Preserve string leading zeroes; never auto-pad numeric source values.'''
    text, error = clean_text(value)
    if error or text is None:
        return text, error
    if re.fullmatch(r"[+-]?\d+(?:\.0+)?", text):
        if "." in text:
            text = text.split(".", 1)[0]
        if text.startswith("+"):
            text = text[1:]
    if not text.isdigit():
        return None, "NOT_DIGITS"
    allowed = {lengths} if isinstance(lengths, int) else set(lengths)
    if len(text) not in allowed:
        return None, "INVALID_LENGTH"
    return text, None


def valid_uuid_text(value):
    text, _ = clean_text(value, 36)
    if text is None:
        return None
    try:
        return str(uuid.UUID(text))
    except (ValueError, AttributeError, TypeError):
        return None


def normalized_status(value):
    text, _ = clean_text(value, 20)
    return text.upper() if text else None


def validate_last_update(value):
    text, error = clean_text(value, 14)
    if error or text is None:
        return text, error
    if not re.fullmatch(r"\d{14}", text):
        return text, "INVALID_LAST_UPD_D_FORMAT"
    try:
        _datetime.datetime.strptime(text, "%Y%m%d%H%M%S")
    except ValueError:
        return text, "INVALID_LAST_UPD_D_VALUE"
    return text, None


def validate_utm(utm1_raw, utm2_raw, utm3_raw, utm4_raw, scale_raw, land_no_raw):
    '''Return canonical APID components and review reasons.'''
    reasons = []

    utm1, error = digit_text(utm1_raw, 4)
    if error or utm1 in (None, "0000"):
        reasons.append("INVALID_UTM1")

    utm2, error = integer_value(utm2_raw)
    if error or utm2 not in {1, 2, 3, 4}:
        reasons.append("INVALID_UTM2")

    utm3, error = digit_text(utm3_raw, 4)
    if error or utm3 is None:
        reasons.append("INVALID_UTM3")

    utm4, error = digit_text(utm4_raw, {2, 3})
    if error or utm4 is None:
        reasons.append("INVALID_UTM4")

    scale, error = integer_value(scale_raw)
    if error or scale not in ALLOWED_SCALES:
        reasons.append("INVALID_UTMSCALE")

    land_no, error = clean_text(land_no_raw, 20)
    if error:
        reasons.append("LAND_NO_OVERFLOW")
    elif land_no is None or land_no == "0":
        reasons.append("INVALID_LAND_NO")

    # Cross-field map-sheet rules. Apply only when the individual values parsed.
    if scale in ALLOWED_SCALES and utm3 is not None:
        if scale == 5000 and utm3 != "0000":
            reasons.append("UTM3_SCALE_MISMATCH")
        elif scale != 5000 and utm3 == "0000":
            reasons.append("UTM3_SCALE_MISMATCH")

    if scale in ALLOWED_SCALES and utm4 is not None:
        if scale == 5000:
            if len(utm4) != 3:
                reasons.append("UTM4_SCALE_MISMATCH")
        elif len(utm4) != 2:
            reasons.append("UTM4_SCALE_MISMATCH")
        else:
            number = int(utm4)
            limits = {4000: (0, 0), 2000: (1, 4), 1000: (1, 16), 500: (1, 64)}
            lower, upper = limits[scale]
            if not lower <= number <= upper:
                reasons.append("UTM4_SCALE_MISMATCH")

    return {
        "UTM1": utm1,
        "UTM2": utm2,
        "UTM3": utm3,
        "UTM4": utm4,
        "UTMSCALE": scale,
        "LAND_NO": land_no,
    }, sorted(set(reasons))


def canonical_attribute_hash(record):
    values = [record.get(name) for name in ATTR_HASH_FIELDS]
    payload = json.dumps(values, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _grid_integer(value, resolution):
    return int((Decimal(str(value)) / Decimal(str(resolution))).to_integral_value(rounding=ROUND_HALF_UP))


def _minimal_rotation(sequence):
    '''Lexicographically minimal cyclic rotation using Booth's algorithm.'''
    if not sequence:
        return tuple()
    doubled = sequence + sequence
    length = len(sequence)
    left, right, offset = 0, 1, 0
    while left < length and right < length and offset < length:
        a = doubled[left + offset]
        b = doubled[right + offset]
        if a == b:
            offset += 1
            continue
        if a > b:
            left = left + offset + 1
            if left <= right:
                left = right + 1
        else:
            right = right + offset + 1
            if right <= left:
                right = left + 1
        offset = 0
    start = min(left, right)
    return tuple(doubled[start:start + length])


def canonical_geometry_hash(shape, expected_wkid):
    '''Hash normalized 2D Esri JSON rings.

    Policy v1:
    - geometry must already be a valid 2D polygon in the expected WKID;
    - XY is snapped to the spatial reference XY resolution;
    - duplicate closing and consecutive duplicate vertices are removed;
    - each ring is rotated to a stable start vertex without reversing direction;
    - ring direction is retained and rings are sorted before serialization.

    A later spatial-equivalence decision must still use geometry comparison; the
    hash is the contract's fast comparison mechanism, not the final lineage test.
    '''
    if shape is None or shape.pointCount == 0:
        raise ValueError("EMPTY_GEOMETRY")

    geometry_json = json.loads(shape.JSON)
    if "curveRings" in geometry_json:
        raise ValueError("UNSUPPORTED_CURVE_GEOMETRY")
    rings = geometry_json.get("rings")
    if not rings:
        raise ValueError("NO_POLYGON_RINGS")

    spatial_reference = shape.spatialReference
    wkid = int(getattr(spatial_reference, "factoryCode", 0) or 0)
    if wkid != expected_wkid:
        raise ValueError(f"ROW_WKID_MISMATCH_{wkid}")

    resolution = float(getattr(spatial_reference, "XYResolution", 0.0001) or 0.0001)
    normalized_rings = []

    for ring in rings:
        points = [(_grid_integer(point[0], resolution), _grid_integer(point[1], resolution)) for point in ring]
        compact = []
        for point in points:
            if not compact or point != compact[-1]:
                compact.append(point)
        if len(compact) > 1 and compact[0] == compact[-1]:
            compact.pop()
        if len(set(compact)) < 3:
            raise ValueError("DEGENERATE_RING")

        area2 = sum(
            compact[index][0] * compact[(index + 1) % len(compact)][1]
            - compact[(index + 1) % len(compact)][0] * compact[index][1]
            for index in range(len(compact))
        )
        if area2 == 0:
            raise ValueError("ZERO_AREA_RING")
        direction = 1 if area2 > 0 else -1
        normalized_rings.append((direction, _minimal_rotation(tuple(compact))))

    normalized_rings.sort()
    payload = {
        "policy": "CANONICAL_RING_V1",
        "wkid": wkid,
        "xy_resolution": format(resolution, ".15g"),
        "rings": normalized_rings,
    }
    encoded = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(encoded.encode("utf-8")).hexdigest()

## 4. Inventory and source selection

In [4]:
def parse_feature_class_name(name):
    match = FC_PATTERN.fullmatch(name)
    if not match:
        return None
    branch = match.group("branch")
    zone = int(match.group("zone"))
    return {
        "source_name": name,
        "branch_code": branch,
        "prov_id": branch[:2],
        "zone": zone,
        "expected_wkid": EXPECTED_WKID[zone],
        "final_c": f"C{branch}_{zone}",
        "final_z": f"Z{branch}_{zone}",
        "temp_c": f"TC{branch}_{zone}",
        "temp_z": f"TZ{branch}_{zone}",
        "target_gdb_name": f"P{branch[:2]}.gdb",
    }


def field_lookup(feature_class):
    return {field.name.upper(): field.name for field in arcpy.ListFields(feature_class)}


def schema_fingerprint(feature_class):
    schema = [
        (field.name.upper(), field.type, field.length, field.isNullable)
        for field in arcpy.ListFields(feature_class)
    ]
    payload = json.dumps(schema, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def inventory_sources():
    if not arcpy.Exists(as_path(PREPARE_GDB)):
        raise FileNotFoundError(f"PREPARE_GDB not found: {PREPARE_GDB}")

    inventory = []
    ignored = []
    seen_names = set()

    for directory, _, feature_classes in arcpy.da.Walk(as_path(PREPARE_GDB), datatype="FeatureClass"):
        for feature_class_name in feature_classes:
            source_path = os.path.join(directory, feature_class_name)
            parsed = parse_feature_class_name(feature_class_name)
            if parsed is None:
                ignored.append(feature_class_name)
                continue
            if feature_class_name.upper() in seen_names:
                raise RuntimeError(f"Duplicate feature class name found: {feature_class_name}")
            seen_names.add(feature_class_name.upper())

            description = arcpy.Describe(source_path)
            source_fields = field_lookup(source_path)
            row_count = int(arcpy.management.GetCount(source_path)[0])
            wkid = int(getattr(description.spatialReference, "factoryCode", 0) or 0)

            errors = []
            if str(description.shapeType).upper() != "POLYGON":
                errors.append("NOT_POLYGON")
            if wkid != parsed["expected_wkid"]:
                errors.append(f"WKID_MISMATCH_{wkid}")
            if not ALLOW_Z_OR_M and (bool(description.hasZ) or bool(description.hasM)):
                errors.append("Z_OR_M_GEOMETRY_NOT_SUPPORTED")
            for required in ("LANDOFFICE", "CHANGWAT_CODE"):
                if required not in source_fields:
                    errors.append(f"MISSING_REQUIRED_FIELD_{required}")

            missing_conditional = [
                name for name in ("UTM1", "UTM2", "UTM3", "UTM4", "UTMSCALE", "LAND_NO")
                if name not in source_fields
            ]

            inventory.append({
                **parsed,
                "source_path": source_path,
                "source_count": row_count,
                "source_wkid": wkid,
                "has_z": bool(description.hasZ),
                "has_m": bool(description.hasM),
                "schema_fingerprint": schema_fingerprint(source_path),
                "missing_conditional_fields": ",".join(missing_conditional),
                "preflight_status": "OK" if not errors else "ERROR",
                "preflight_errors": ";".join(errors),
            })

    inventory.sort(key=lambda item: (item["prov_id"], item["branch_code"], item["zone"]))
    return inventory, sorted(ignored)

## 5. Versioned SQLite checkpoints and reports

In [5]:
class CheckpointConflict(RuntimeError):
    """An existing checkpoint belongs to incompatible inputs or export rules."""


def normalized_path(path):
    return os.path.normcase(os.path.abspath(as_path(path)))


def policy_fingerprint():
    """Fingerprint settings affecting row interpretation; version covers code."""
    contract = {
        "policy_version": EXPORT_POLICY_VERSION,
        "canonical_fields": CANONICAL_FIELDS,
        "attribute_hash_fields": ATTR_HASH_FIELDS,
        "valid_statuses": sorted(VALID_STATUSES),
        "allowed_scales": sorted(ALLOWED_SCALES),
        "expected_wkid": EXPECTED_WKID,
        "check_geometry": RUN_CHECK_GEOMETRY,
        "allow_z_or_m": ALLOW_Z_OR_M,
        "geometry_hash_policy": "CANONICAL_RING_V1",
    }
    return hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()


def state_context():
    return {
        "policy_version": EXPORT_POLICY_VERSION,
        "policy_fingerprint": policy_fingerprint(),
        "acquired": ACQUIRED,
        "prepare_gdb": normalized_path(PREPARE_GDB),
        "archive_dir": normalized_path(ARCHIVE_DIR),
    }


def open_state_database():
    """Open V2 state; reject an old/foreign database without rewriting it."""
    existed = STATE_DB.exists()
    if not existed:
        REPORT_DIR.mkdir(parents=True, exist_ok=True)
    connection = sqlite3.connect(as_path(STATE_DB), timeout=60)
    connection.row_factory = sqlite3.Row
    try:
        if existed:
            schema_version = connection.execute("PRAGMA user_version").fetchone()[0]
            if schema_version != STATE_SCHEMA_VERSION:
                raise CheckpointConflict(
                    f"State schema {schema_version} is incompatible with V{STATE_SCHEMA_VERSION}. "
                    "Keep the previous state; choose fresh V2 state and staging paths."
                )
            try:
                stored = dict(connection.execute("SELECT key, value FROM export_metadata"))
            except sqlite3.DatabaseError as error:
                raise CheckpointConflict("State lacks V2 export metadata") from error
            changed = [key for key, value in state_context().items() if stored.get(key) != value]
            if changed:
                raise CheckpointConflict(
                    "State context differs: " + ", ".join(changed)
                    + ". Use fresh state and a fresh staging directory."
                )

        connection.execute("PRAGMA journal_mode=WAL")
        connection.execute("PRAGMA synchronous=FULL")
        if not existed:
            connection.executescript('''
                CREATE TABLE export_metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL);
                CREATE TABLE feature_state (
                    acquired TEXT NOT NULL, source_name TEXT NOT NULL,
                    source_path TEXT NOT NULL, branch_code TEXT NOT NULL,
                    prov_id TEXT NOT NULL, zone INTEGER NOT NULL,
                    expected_wkid INTEGER NOT NULL, schema_fingerprint TEXT NOT NULL,
                    policy_version TEXT NOT NULL, policy_fingerprint TEXT NOT NULL,
                    source_count INTEGER NOT NULL,
                    c_count INTEGER, z_count INTEGER, rejected_count INTEGER,
                    target_gdb TEXT NOT NULL, final_c TEXT NOT NULL, final_z TEXT NOT NULL,
                    temp_c TEXT NOT NULL, temp_z TEXT NOT NULL, status TEXT NOT NULL,
                    started_at TEXT, finished_at TEXT, error_message TEXT,
                    PRIMARY KEY (acquired, source_name)
                );
                CREATE TABLE validation_counts (
                    acquired TEXT NOT NULL, source_name TEXT NOT NULL,
                    outcome TEXT NOT NULL, rule TEXT NOT NULL, row_count INTEGER NOT NULL,
                    PRIMARY KEY (acquired, source_name, outcome, rule)
                );
                CREATE TABLE rejected_rows (
                    acquired TEXT NOT NULL, source_name TEXT NOT NULL,
                    source_oid TEXT NOT NULL, reasons TEXT NOT NULL,
                    PRIMARY KEY (acquired, source_name, source_oid)
                );
                CREATE TABLE province_counts (
                    acquired TEXT NOT NULL, source_name TEXT NOT NULL,
                    prov_id TEXT NOT NULL, changwat_code TEXT NOT NULL,
                    outcome TEXT NOT NULL, row_count INTEGER NOT NULL,
                    PRIMARY KEY (acquired, source_name, prov_id, changwat_code, outcome)
                );
                CREATE TABLE run_errors (
                    error_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    acquired TEXT NOT NULL, source_name TEXT NOT NULL,
                    occurred_at TEXT NOT NULL, checkpoint_status TEXT,
                    error_message TEXT NOT NULL, resolved_at TEXT
                );
            ''')
            with connection:
                connection.executemany(
                    "INSERT INTO export_metadata VALUES (?, ?)", state_context().items()
                )
                connection.execute(f"PRAGMA user_version = {STATE_SCHEMA_VERSION}")
        return connection
    except BaseException:
        connection.close()
        raise


def get_feature_state(connection, source_name):
    return connection.execute(
        "SELECT * FROM feature_state WHERE acquired = ? AND source_name = ?",
        (ACQUIRED, source_name),
    ).fetchone()


def upsert_feature_state(connection, item, status, **values):
    """Update only compatible state; a mismatch must never rebase its metadata."""
    current = get_feature_state(connection, item["source_name"])
    if current:
        validate_resume_source(current, item)
    base = {
        "acquired": ACQUIRED,
        "source_name": item["source_name"],
        "source_path": normalized_path(item["source_path"]),
        "branch_code": item["branch_code"], "prov_id": item["prov_id"],
        "zone": item["zone"], "expected_wkid": item["expected_wkid"],
        "schema_fingerprint": item["schema_fingerprint"],
        "policy_version": EXPORT_POLICY_VERSION,
        "policy_fingerprint": policy_fingerprint(),
        "source_count": item["source_count"],
        "c_count": current["c_count"] if current else None,
        "z_count": current["z_count"] if current else None,
        "rejected_count": current["rejected_count"] if current else None,
        "target_gdb": normalized_path(ARCHIVE_DIR / item["target_gdb_name"]),
        "final_c": item["final_c"], "final_z": item["final_z"],
        "temp_c": item["temp_c"], "temp_z": item["temp_z"],
        "status": status,
        "started_at": current["started_at"] if current else None,
        "finished_at": current["finished_at"] if current else None,
        "error_message": None,
    }
    base.update(values)
    columns = list(base)
    placeholders = ",".join("?" for _ in columns)
    updates = ",".join(f"{column}=excluded.{column}" for column in columns[2:])
    with connection:
        connection.execute(
            f"INSERT INTO feature_state ({','.join(columns)}) VALUES ({placeholders}) "
            f"ON CONFLICT(acquired, source_name) DO UPDATE SET {updates}",
            [base[column] for column in columns],
        )


def flush_rejected_rows(connection, source_name, rows):
    """Flush bounded diagnostics; these are not a completed-export checkpoint."""
    if rows:
        with connection:
            connection.executemany(
                "INSERT INTO rejected_rows VALUES (?, ?, ?, ?)",
                ((ACQUIRED, source_name, str(oid), ";".join(reasons)) for oid, reasons in rows),
            )
        rows.clear()


def replace_validation_results(connection, item, counters, province_counters):
    source_name = item["source_name"]
    with connection:
        for table in ("validation_counts", "province_counts"):
            connection.execute(
                f"DELETE FROM {table} WHERE acquired = ? AND source_name = ?",
                (ACQUIRED, source_name),
            )
        connection.executemany(
            "INSERT INTO validation_counts VALUES (?, ?, ?, ?, ?)",
            [(ACQUIRED, source_name, outcome, rule, count)
             for (outcome, rule), count in sorted(counters.items())],
        )
        connection.executemany(
            "INSERT INTO province_counts VALUES (?, ?, ?, ?, ?, ?)",
            [(ACQUIRED, source_name, item["prov_id"], changwat, outcome, count)
             for (changwat, outcome), count in sorted(province_counters.items())],
        )


def province_report_key(values):
    """Bound reporting cardinality; placeholder labels exist only in reports."""
    code, error = digit_text(values.get("CHANGWAT_CODE"), 2)
    return "<INVALID>" if error else (code if code is not None else "<MISSING>")


def record_run_error(connection, item, error_message):
    current = get_feature_state(connection, item["source_name"])
    with connection:
        connection.execute(
            "INSERT INTO run_errors "
            "(acquired, source_name, occurred_at, checkpoint_status, error_message) "
            "VALUES (?, ?, ?, ?, ?)",
            (ACQUIRED, item["source_name"], utc_now(),
             current["status"] if current else None, error_message),
        )


def resolve_run_errors(connection, source_name):
    with connection:
        connection.execute(
            "UPDATE run_errors SET resolved_at = ? WHERE acquired = ? "
            "AND source_name = ? AND resolved_at IS NULL",
            (utc_now(), ACQUIRED, source_name),
        )


def dataframe_from_query(connection, sql, parameters=()):
    return pd.read_sql_query(sql, connection, params=parameters)


def write_excel_report(connection):
    """Regenerate a bounded Excel review from durable SQLite state."""
    parameters = (ACQUIRED,)
    feature_df = dataframe_from_query(
        connection,
        "SELECT * FROM feature_state WHERE acquired = ? ORDER BY prov_id, branch_code, zone",
        parameters,
    )
    validation_df = dataframe_from_query(
        connection,
        "SELECT * FROM validation_counts WHERE acquired = ? ORDER BY source_name, outcome, rule",
        parameters,
    )
    rejected_df = dataframe_from_query(
        connection,
        "SELECT * FROM rejected_rows WHERE acquired = ? ORDER BY source_name, source_oid LIMIT ?",
        (ACQUIRED, REJECTED_PREVIEW_LIMIT),
    )
    rejection_summary_df = dataframe_from_query(
        connection,
        "SELECT r.source_name, f.status AS checkpoint_status, r.reasons, COUNT(*) AS row_count "
        "FROM rejected_rows r LEFT JOIN feature_state f "
        "ON r.acquired = f.acquired AND r.source_name = f.source_name "
        "WHERE r.acquired = ? GROUP BY r.source_name, f.status, r.reasons "
        "ORDER BY r.source_name, row_count DESC", parameters,
    )
    admin_df = dataframe_from_query(
        connection,
        "SELECT p.source_name, f.status AS checkpoint_status, p.prov_id AS PROV_ID, "
        "p.changwat_code AS CHANGWAT_CODE, p.outcome, p.row_count, "
        "CASE WHEN p.changwat_code IN ('<INVALID>', '<MISSING>') THEN NULL "
        "WHEN p.prov_id <> p.changwat_code THEN 1 ELSE 0 END AS SERVICE_ADMIN_DIFFERENT "
        "FROM province_counts p JOIN feature_state f "
        "ON p.acquired = f.acquired AND p.source_name = f.source_name "
        "WHERE p.acquired = ? ORDER BY p.source_name, p.changwat_code, p.outcome", parameters,
    )
    run_errors_df = dataframe_from_query(
        connection, "SELECT * FROM run_errors WHERE acquired = ? ORDER BY error_id", parameters
    )
    errors_df = run_errors_df[run_errors_df["resolved_at"].isna()].copy()
    total_rejected = connection.execute(
        "SELECT COUNT(*) FROM rejected_rows WHERE acquired = ?", parameters
    ).fetchone()[0]
    report_info_df = pd.DataFrame([{
        "POLICY_VERSION": EXPORT_POLICY_VERSION,
        "STATE_DB": as_path(STATE_DB), "ARCHIVE_DIR": as_path(ARCHIVE_DIR),
        "DETAIL_TABLE": "rejected_rows", "ACQUIRED_FILTER": ACQUIRED,
        "DETAIL_ROWS_STORED": total_rejected, "PREVIEW_ROWS": len(rejected_df),
        "PREVIEW_TRUNCATED": total_rejected > len(rejected_df),
        "NOTE": (
            "PROV_ID is service province; CHANGWAT_CODE is administrative province. "
            "Different provinces are informational. Failed/BUILDING diagnostics may be partial. "
            "Completed totals describe stored checkpoints; inspect ERRORS for failed current verification."
        ),
    }])
    completed = feature_df[feature_df["status"] == "COMPLETED"]
    summary_df = pd.DataFrame([{
        "ACQUIRED": ACQUIRED, "POLICY_VERSION": EXPORT_POLICY_VERSION,
        "FEATURE_CLASSES": len(feature_df), "COMPLETED_CHECKPOINTS": len(completed),
        "FAILED": int((feature_df["status"] == "FAILED").sum()),
        "UNRESOLVED_RUN_ERRORS": len(errors_df),
        "SOURCE_ROWS_COMPLETED": int(completed["source_count"].fillna(0).sum()),
        "C_ROWS": int(completed["c_count"].fillna(0).sum()),
        "Z_ROWS": int(completed["z_count"].fillna(0).sum()),
        "REJECTED_ROWS": int(completed["rejected_count"].fillna(0).sum()),
        "REPORT_GENERATED_UTC": utc_now(),
    }])
    if completed.empty:
        gdb_df = pd.DataFrame(columns=[
            "PROV_ID", "target_gdb", "feature_classes", "source_rows", "c_rows", "z_rows", "rejected_rows"
        ])
    else:
        gdb_df = completed.groupby(["prov_id", "target_gdb"], dropna=False).agg(
            feature_classes=("source_name", "count"), source_rows=("source_count", "sum"),
            c_rows=("c_count", "sum"), z_rows=("z_count", "sum"),
            rejected_rows=("rejected_count", "sum"),
        ).reset_index().rename(columns={"prov_id": "PROV_ID"})

    sheets = {
        "RUN_SUMMARY": summary_df, "FEATURE_CLASSES": feature_df,
        "GDB_SUMMARY": gdb_df, "ADMIN_PROVINCE_SUMMARY": admin_df,
        "VALIDATION_SUMMARY": validation_df, "REJECTION_SUMMARY": rejection_summary_df,
        "REJECTED_PREVIEW": rejected_df, "REPORT_INFO": report_info_df,
        "ERRORS": errors_df, "RUN_ERRORS": run_errors_df,
    }
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    temporary_report = EXPORT_REPORT.with_name(f".{EXPORT_REPORT.stem}.{uuid.uuid4().hex}.tmp.xlsx")
    try:
        with pd.ExcelWriter(temporary_report, engine="openpyxl") as writer:
            for name, frame in sheets.items():
                if len(frame) > 1_048_575:
                    raise RuntimeError(f"{name} exceeds Excel's row limit; use SQLite for full details")
                frame.to_excel(writer, sheet_name=name, index=False)
            for worksheet in writer.book.worksheets:
                worksheet.freeze_panes = "A2"
                worksheet.auto_filter.ref = worksheet.dimensions
                for column in worksheet.iter_cols(max_row=min(worksheet.max_row, 200)):
                    values = [str(cell.value) for cell in column if cell.value is not None]
                    width = min(max([len(value) for value in values] + [10]) + 2, 60)
                    worksheet.column_dimensions[column[0].column_letter].width = width
        os.replace(temporary_report, EXPORT_REPORT)
    finally:
        if temporary_report.exists():
            temporary_report.unlink()
    print(f"Report updated: {EXPORT_REPORT}")


def try_write_excel_report(connection):
    try:
        write_excel_report(connection)
        return True
    except Exception as error:
        print(f"Report could not be updated ({type(error).__name__}): {error}. "
              f"Inspect SQLite state at {STATE_DB}; export status is unchanged.")
        return False

## 6. Geodatabase schema and publication

In [6]:
def add_fields(dataset, field_specs):
    for name, field_type, length, nullable in field_specs:
        arcpy.management.AddField(
            dataset,
            name,
            field_type,
            field_length=length,
            field_is_nullable="NULLABLE" if nullable else "NON_NULLABLE",
        )


def ensure_archive_gdb(item):
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    target_gdb = ARCHIVE_DIR / item["target_gdb_name"]
    if not arcpy.Exists(as_path(target_gdb)):
        arcpy.management.CreateFileGDB(as_path(ARCHIVE_DIR), item["target_gdb_name"])
        print(f"Created {target_gdb}")

    relationship_table = os.path.join(as_path(target_gdb), "ENTITY_RELATIONSHIP")
    if not arcpy.Exists(relationship_table):
        arcpy.management.CreateTable(as_path(target_gdb), "ENTITY_RELATIONSHIP")
        add_fields(relationship_table, RELATIONSHIP_FIELDS)
    return target_gdb


def canonical_field_signature(dataset):
    fields = {field.name.upper(): field for field in arcpy.ListFields(dataset)}
    return {
        name: (fields[name].type.upper(), fields[name].length, fields[name].isNullable)
        for name in CANONICAL_NAMES
        if name in fields
    }


def validate_canonical_schema(dataset):
    fields = {field.name.upper(): field for field in arcpy.ListFields(dataset)}
    missing = [name for name in CANONICAL_NAMES if name not in fields]
    if missing:
        raise RuntimeError(f"Canonical fields missing from {dataset}: {missing}")
    type_names = {"TEXT": "STRING", "LONG": "INTEGER", "DOUBLE": "DOUBLE"}
    problems = []
    for name, field_type, length, nullable in CANONICAL_FIELDS:
        field = fields[name]
        if field.type.upper() != type_names[field_type]:
            problems.append(f"{name}: type {field.type}")
        if field_type == "TEXT" and field.length != length:
            problems.append(f"{name}: length {field.length}, expected {length}")
        if bool(field.isNullable) != nullable:
            problems.append(f"{name}: nullable={field.isNullable}, expected {nullable}")
    if problems:
        raise RuntimeError(f"Canonical schema mismatch in {dataset}: {problems}")


def create_temp_feature_class(target_gdb, name, spatial_reference):
    output = os.path.join(as_path(target_gdb), name)
    if arcpy.Exists(output):
        arcpy.management.Delete(output)
    arcpy.management.CreateFeatureclass(
        as_path(target_gdb),
        name,
        "POLYGON",
        spatial_reference=spatial_reference,
        has_m="DISABLED",
        has_z="DISABLED",
    )
    add_fields(output, CANONICAL_FIELDS)
    validate_canonical_schema(output)
    return output


def ensure_work_gdb():
    '''Create a dedicated writable workspace for Check Geometry results.

    ArcGIS Pro's automatic scratch.gdb can be unavailable or locked in a
    notebook session, which raises ERROR 000210. Keeping these temporary tables
    under REPORT_DIR makes the location explicit and restart-safe.
    '''
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    if not arcpy.Exists(as_path(WORK_GDB)):
        arcpy.management.CreateFileGDB(as_path(REPORT_DIR), WORK_GDB.name)
    if not arcpy.Exists(as_path(WORK_GDB)):
        raise RuntimeError(f"Could not create work geodatabase: {WORK_GDB}")
    return as_path(WORK_GDB)


def geometry_problem_map(source_path, source_name):
    '''Return {source OID: sorted problem codes}. PREPARE_GDB is not repaired.'''
    if not RUN_CHECK_GEOMETRY:
        return {}
    work_gdb = ensure_work_gdb()
    table_name = arcpy.ValidateTableName(f"CHK_{source_name}", work_gdb)
    output_table = os.path.join(work_gdb, table_name)
    if arcpy.Exists(output_table):
        arcpy.management.Delete(output_table)
    problems = defaultdict(set)
    try:
        arcpy.management.CheckGeometry(source_path, output_table)
        table_fields = {field.name.upper(): field.name for field in arcpy.ListFields(output_table)}
        feature_id_field = table_fields.get("FEATURE_ID")
        problem_field = table_fields.get("PROBLEM")
        if not feature_id_field or not problem_field:
            raise RuntimeError("CheckGeometry output lacks FEATURE_ID/PROBLEM fields")
        if feature_id_field and problem_field:
            with arcpy.da.SearchCursor(output_table, [feature_id_field, problem_field]) as rows:
                for feature_id, problem in rows:
                    problems[feature_id].add(str(problem))
    finally:
        if arcpy.Exists(output_table):
            arcpy.management.Delete(output_table)
    return {oid: sorted(values) for oid, values in problems.items()}


def output_count(path):
    return int(arcpy.management.GetCount(path)[0]) if arcpy.Exists(path) else None


def verify_expected_output(path, expected_count):
    exists = arcpy.Exists(path)
    if expected_count == 0:
        if exists:
            raise RuntimeError(f"Expected no output, but dataset exists: {path}")
        return
    if not exists:
        raise RuntimeError(f"Expected output is missing: {path}")
    actual_count = output_count(path)
    if actual_count != expected_count:
        raise RuntimeError(f"Count mismatch for {path}: expected {expected_count:,}, found {actual_count:,}")
    validate_canonical_schema(path)


def publish_one(temp_path, final_path, expected_count):
    temp_exists = arcpy.Exists(temp_path)
    final_exists = arcpy.Exists(final_path)

    if expected_count == 0:
        if final_exists:
            raise RuntimeError(f"Unexpected existing empty-output target: {final_path}")
        if temp_exists:
            arcpy.management.Delete(temp_path)
        return

    # Supports a crash after Rename but before the checkpoint was advanced.
    if final_exists and not temp_exists:
        verify_expected_output(final_path, expected_count)
        return
    if final_exists and temp_exists:
        raise RuntimeError(f"Both temporary and final outputs exist: {temp_path}, {final_path}")
    if not temp_exists:
        raise RuntimeError(f"Temporary output is missing: {temp_path}")

    if output_count(temp_path) != expected_count:
        raise RuntimeError(f"Temporary output count mismatch: {temp_path}")
    arcpy.management.Rename(temp_path, final_path, "FeatureClass")
    verify_expected_output(final_path, expected_count)

## 7. Transform rows with separate administrative province

In [7]:
def canonical_optional_text(value, name, max_length, review_reasons, fatal_reasons):
    normalized, error = clean_text(value, max_length)
    if error:
        fatal_reasons.append(f"{name}_{error}")
    return normalized


def canonical_optional_long(value, name, review_reasons):
    normalized, error = integer_value(value)
    if error:
        review_reasons.append(f"INVALID_{name}")
    return normalized


def transform_row(values, item, geometry_problems):
    '''Return (outcome, canonical_record, reasons).

    outcome is C, Z, or REJECT. The source row is never modified.
    '''
    source_oid = values["OID@"]
    shape = values["SHAPE@"]
    fatal = []
    review = []

    if geometry_problems:
        fatal.extend(f"GEOMETRY_{problem}" for problem in geometry_problems)
    if shape is None or shape.pointCount == 0:
        fatal.append("EMPTY_GEOMETRY")

    source_branch, branch_error = digit_text(values.get("BRANCH_CODE"), 8)
    if values.get("BRANCH_CODE") not in (None, "") and (branch_error or source_branch != item["branch_code"]):
        fatal.append("BRANCH_CODE_NAME_MISMATCH")

    source_changwat, changwat_error = digit_text(values.get("CHANGWAT_CODE"), 2)
    # Administrative province is independent of the office service province.
    # Keep the non-nullable canonical contract: never invent an administrative
    # code from the branch prefix when the source value is missing/invalid.
    if changwat_error:
        fatal.append("INVALID_CHANGWAT_CODE")
    elif source_changwat is None:
        fatal.append("MISSING_CHANGWAT_CODE")

    landoffice, landoffice_error = integer_value(values.get("LANDOFFICE"))
    if landoffice_error or landoffice is None or landoffice <= 0:
        fatal.append("INVALID_LANDOFFICE")

    record = {
        "CHANGWAT_CODE": source_changwat,
        "BRANCH_CODE": item["branch_code"],
        "LANDOFFICE": landoffice,
    }

    amphoe, error = digit_text(values.get("AMPHOE_CODE"), 4)
    if error:
        review.append("INVALID_AMPHOE_CODE")
    elif amphoe and source_changwat and not amphoe.startswith(source_changwat):
        review.append("AMPHOE_CHANGWAT_MISMATCH")
    record["AMPHOE_CODE"] = amphoe

    tambon, error = digit_text(values.get("TAMBON_CODE"), 6)
    if error:
        review.append("INVALID_TAMBON_CODE")
    elif tambon and source_changwat and not tambon.startswith(source_changwat):
        review.append("TAMBON_CHANGWAT_MISMATCH")
    if tambon and amphoe and not tambon.startswith(amphoe):
        review.append("TAMBON_AMPHOE_MISMATCH")
    record["TAMBON_CODE"] = tambon

    record["RAWANG_NAME"] = canonical_optional_text(
        values.get("RAWANG_NAME"), "RAWANG_NAME", 200, review, fatal
    )

    utm_values, utm_reasons = validate_utm(
        values.get("UTM1"), values.get("UTM2"), values.get("UTM3"),
        values.get("UTM4"), values.get("UTMSCALE"), values.get("LAND_NO"),
    )
    record.update(utm_values)
    review.extend(utm_reasons)

    record["PARCEL_TYPE"] = canonical_optional_long(values.get("PARCEL_TYPE"), "PARCEL_TYPE", review)
    record["DOC_TYPE"] = canonical_optional_long(values.get("DOC_TYPE"), "DOC_TYPE", review)

    last_update, error = validate_last_update(values.get("LAST_UPD_D"))
    if error:
        review.append(error)
    record["LAST_UPD_D"] = last_update
    record["REMARK"] = canonical_optional_text(values.get("REMARK"), "REMARK", 250, review, fatal)

    source_status = normalized_status(values.get("RECORD_STATUS"))
    if source_status and source_status not in VALID_STATUSES:
        review.append("INVALID_SOURCE_RECORD_STATUS")

    if fatal:
        return "REJECT", None, sorted(set(fatal + review))

    try:
        label_point = shape.labelPoint  # already a Point with X/Y; it has no .firstPoint
        record["INSIDE_X"] = float(label_point.X)
        record["INSIDE_Y"] = float(label_point.Y)
        record["GEOM_HASH"] = canonical_geometry_hash(shape, item["expected_wkid"])
    except (ValueError, ArithmeticError) as error:
        # Only the data-quality errors raised by canonical_geometry_hash become row
        # rejections. Programming errors (AttributeError, TypeError, ...) must propagate
        # so they fail the feature class instead of silently rejecting every row.
        return "REJECT", None, sorted(set(review + [f"GEOMETRY_SERIALIZATION_{error}"]))

    # APID exists only if all scale-aware UTM checks passed.
    if not utm_reasons:
        record["APID"] = (
            f"{record['LANDOFFICE']}-{record['UTM1']}{record['UTM2']}-"
            f"{record['UTM3']}-{record['UTMSCALE']}-{record['UTM4']}-{record['LAND_NO']}"
        )
        if len(record["APID"]) > 68:
            return "REJECT", None, sorted(set(review + ["APID_OVERFLOW_68"]))
    else:
        record["APID"] = None

    record["ATTR_HASH"] = canonical_attribute_hash(record)
    record["ACQUIRED"] = ACQUIRED

    existing_entity_id = valid_uuid_text(values.get("ENTITY_ID"))
    existing_version_id = valid_uuid_text(values.get("VERSION_ID"))

    # Preserve deliberate valid non-active source states. ACTIVE is re-derived.
    if source_status in NON_ACTIVE_STATUSES:
        record_status = source_status
        outcome = "Z"
    elif review:
        record_status = "PENDING_REVIEW"
        outcome = "Z"
    else:
        record_status = "ACTIVE"
        outcome = "C"
    record["RECORD_STATUS"] = record_status

    # ENTITY_ID is required for ACTIVE/SUPERSEDED/RETIRED and optional otherwise.
    if existing_entity_id:
        record["ENTITY_ID"] = existing_entity_id
    elif record_status in {"ACTIVE", "SUPERSEDED", "RETIRED"}:
        record["ENTITY_ID"] = str(uuid.uuid5(
            UUID_NAMESPACE,
            f"{ACQUIRED}|{item['source_name']}|{source_oid}|ENTITY",
        ))
    else:
        record["ENTITY_ID"] = None

    record["VERSION_ID"] = existing_version_id or str(uuid.uuid5(
        UUID_NAMESPACE,
        f"{ACQUIRED}|{item['source_name']}|{source_oid}|VERSION|"
        f"{record['GEOM_HASH']}|{record['ATTR_HASH']}",
    ))

    return outcome, record, sorted(set(review))

## 8. Build, publish and resume one feature class

In [8]:
def available_source_cursor_fields(source_path):
    lookup = field_lookup(source_path)
    available = [lookup[name] for name in SOURCE_CANDIDATE_FIELDS if name in lookup]
    canonical_name_by_actual = {lookup[name]: name for name in SOURCE_CANDIDATE_FIELDS if name in lookup}
    return ["OID@", "SHAPE@"] + available, canonical_name_by_actual


def safe_delete_incomplete_temps(target_gdb, item):
    """Delete only this source's temporary datasets after ownership checks."""
    for name in (item["temp_c"], item["temp_z"]):
        path = os.path.join(as_path(target_gdb), name)
        if arcpy.Exists(path):
            arcpy.management.Delete(path)


def ensure_no_ambiguous_finals(target_gdb, item):
    if OVERWRITE_EXISTING:
        raise RuntimeError("OVERWRITE_EXISTING must remain False in BOOTSTRAP mode")
    existing = [
        os.path.join(as_path(target_gdb), name)
        for name in (item["final_c"], item["final_z"])
        if arcpy.Exists(os.path.join(as_path(target_gdb), name))
    ]
    if existing:
        raise CheckpointConflict(
            "Existing final outputs have no compatible BUILT/COMPLETED checkpoint: "
            + ", ".join(existing) + ". Choose a fresh staging directory."
        )


def validate_resume_source(state, item):
    comparisons = {
        "source_path": normalized_path(item["source_path"]),
        "source_count": item["source_count"],
        "expected_wkid": item["expected_wkid"],
        "schema_fingerprint": item["schema_fingerprint"],
        "branch_code": item["branch_code"], "prov_id": item["prov_id"],
        "zone": item["zone"],
        "policy_version": EXPORT_POLICY_VERSION,
        "policy_fingerprint": policy_fingerprint(),
        "target_gdb": normalized_path(ARCHIVE_DIR / item["target_gdb_name"]),
        "final_c": item["final_c"], "final_z": item["final_z"],
        "temp_c": item["temp_c"], "temp_z": item["temp_z"],
    }
    changed = [name for name, current in comparisons.items()
               if name not in state.keys() or state[name] != current]
    if changed:
        raise CheckpointConflict(
            f"Checkpoint differs for {item['source_name']}: {', '.join(changed)}. "
            "Keep that checkpoint; rebuild with fresh state and a fresh staging directory."
        )


def checkpoint_counts(state):
    """A resumable publication must have a complete, reconciled count record."""
    counts = [state[key] for key in ("c_count", "z_count", "rejected_count")]
    if any(not isinstance(value, int) or value < 0 for value in counts):
        raise CheckpointConflict("Checkpoint lacks valid C/Z/rejected counts")
    if sum(counts) != state["source_count"]:
        raise CheckpointConflict("Checkpoint row counts do not reconcile")
    return counts


def reconcile_and_complete(connection, item, state):
    validate_resume_source(state, item)
    c_count, z_count, _ = checkpoint_counts(state)
    target_gdb = Path(state["target_gdb"])
    final_c = os.path.join(as_path(target_gdb), state["final_c"])
    final_z = os.path.join(as_path(target_gdb), state["final_z"])
    publish_one(os.path.join(as_path(target_gdb), state["temp_c"]), final_c, c_count)
    publish_one(os.path.join(as_path(target_gdb), state["temp_z"]), final_z, z_count)
    upsert_feature_state(connection, item, "PUBLISHED")
    verify_expected_output(final_c, c_count)
    verify_expected_output(final_z, z_count)
    upsert_feature_state(connection, item, "COMPLETED", finished_at=utc_now())


def verify_completed_state(state, item):
    validate_resume_source(state, item)
    c_count, z_count, _ = checkpoint_counts(state)
    target_gdb = Path(state["target_gdb"])
    verify_expected_output(os.path.join(as_path(target_gdb), state["final_c"]), c_count)
    verify_expected_output(os.path.join(as_path(target_gdb), state["final_z"]), z_count)


def build_feature_class(connection, item):
    """Read one immutable source; publish its complete C/Z outputs or fail."""
    source_path = item["source_path"]
    source_name = item["source_name"]
    state = get_feature_state(connection, source_name)
    if state:
        validate_resume_source(state, item)
        if state["status"] == "COMPLETED":
            verify_completed_state(state, item)
            print(f"SKIP completed: {source_name}")
            return
        if state["status"] in {"BUILT", "PUBLISHED"}:
            reconcile_and_complete(connection, item, state)
            print(f"COMPLETED (resumed publication): {source_name}")
            return
        if state["status"] not in {"BUILDING", "FAILED"}:
            raise CheckpointConflict(f"Unsupported checkpoint status: {state['status']}")

    target_gdb = ensure_archive_gdb(item)
    ensure_no_ambiguous_finals(target_gdb, item)
    if state is None:
        unexplained = [name for name in (item["temp_c"], item["temp_z"])
                       if arcpy.Exists(os.path.join(as_path(target_gdb), name))]
        if unexplained:
            raise CheckpointConflict(f"Temporary outputs lack a checkpoint: {unexplained}")
    safe_delete_incomplete_temps(target_gdb, item)
    upsert_feature_state(
        connection, item, "BUILDING", started_at=utc_now(), finished_at=None,
        c_count=None, z_count=None, rejected_count=None,
    )
    # A failed/incomplete feature class is rebuilt in full; its previous
    # diagnostics must not be mixed with those of the replacement attempt.
    with connection:
        for table in ("rejected_rows", "validation_counts", "province_counts"):
            connection.execute(
                f"DELETE FROM {table} WHERE acquired = ? AND source_name = ?",
                (ACQUIRED, source_name),
            )
    description = arcpy.Describe(source_path)
    temp_c = create_temp_feature_class(target_gdb, item["temp_c"], description.spatialReference)
    temp_z = create_temp_feature_class(target_gdb, item["temp_z"], description.spatialReference)
    geometry_problems = geometry_problem_map(source_path, source_name)
    cursor_fields, canonical_name_by_actual = available_source_cursor_fields(source_path)
    insert_fields = ["SHAPE@"] + CANONICAL_NAMES
    counters = Counter()
    province_counters = Counter()
    rejected_rows = []
    processed = c_count = z_count = rejected_count = 0

    # Both insert cursors share the original notebook's single edit session.
    # Cursors close before commit; exceptions roll back the edit session.
    try:
        with arcpy.da.Editor(as_path(target_gdb), multiuser_mode=False):
            with arcpy.da.SearchCursor(source_path, cursor_fields) as source_rows, \
                    arcpy.da.InsertCursor(temp_c, insert_fields) as c_rows, \
                    arcpy.da.InsertCursor(temp_z, insert_fields) as z_rows:
                for source_row in source_rows:
                    values = {"OID@": source_row[0], "SHAPE@": source_row[1]}
                    for actual_name, value in zip(cursor_fields[2:], source_row[2:]):
                        values[canonical_name_by_actual[actual_name]] = value
                    oid = values["OID@"]
                    outcome, record, reasons = transform_row(
                        values, item, geometry_problems.get(oid, [])
                    )
                    # Informational counts are separate from validation reasons:
                    # CHANGWAT_CODE != PROV_ID never turns a valid row into Z.
                    province_counters[(province_report_key(values), outcome)] += 1
                    if outcome == "REJECT":
                        rejected_count += 1
                        reasons = reasons or ["UNSPECIFIED_STRUCTURAL_FAILURE"]
                        rejected_rows.append((oid, reasons))
                        if len(rejected_rows) >= REJECT_BATCH_SIZE:
                            flush_rejected_rows(connection, source_name, rejected_rows)
                    elif outcome in {"C", "Z"}:
                        output_values = [values["SHAPE@"]] + [record.get(name) for name in CANONICAL_NAMES]
                        if outcome == "C":
                            c_rows.insertRow(output_values)
                            c_count += 1
                        else:
                            z_rows.insertRow(output_values)
                            z_count += 1
                    else:
                        raise RuntimeError(f"Unexpected transformation outcome: {outcome}")
                    for reason in reasons or ["ACCEPTED_WITHOUT_VALIDATION_ISSUES"]:
                        counters[(outcome, reason)] += 1
                    processed += 1
                    if PROGRESS_EVERY and processed % PROGRESS_EVERY == 0:
                        print(f"  {source_name}: {processed:,}/{item['source_count']:,} "
                              f"C={c_count:,} Z={z_count:,} rejected={rejected_count:,}")
    finally:
        flush_rejected_rows(connection, source_name, rejected_rows)

    if processed != item["source_count"]:
        raise RuntimeError(f"Source count changed: expected {item['source_count']:,}, read {processed:,}")
    if processed != c_count + z_count + rejected_count:
        raise RuntimeError("Row reconciliation failed")
    if sum(province_counters.values()) != processed:
        raise RuntimeError("Province summary reconciliation failed")
    if output_count(temp_c) != c_count or output_count(temp_z) != z_count:
        raise RuntimeError("Temporary output count verification failed")
    replace_validation_results(connection, item, counters, province_counters)
    # Persist reconciled counts even for an all-rejected failure.
    upsert_feature_state(connection, item, "BUILDING", c_count=c_count,
                         z_count=z_count, rejected_count=rejected_count)
    if item["source_count"] > 0 and c_count + z_count == 0:
        safe_delete_incomplete_temps(target_gdb, item)
        raise RuntimeError(
            f"{source_name}: all {rejected_count:,} rows were rejected; "
            f"nothing is storable. Top reasons: {counters.most_common(3)}"
        )
    upsert_feature_state(connection, item, "BUILT")
    reconcile_and_complete(connection, item, get_feature_state(connection, source_name))
    print(f"COMPLETED: {source_name} | source={processed:,} "
          f"C={c_count:,} Z={z_count:,} rejected={rejected_count:,}")

## 9. Contract smoke checks

In [9]:
def run_contract_smoke_tests():
    """Run small deterministic contract checks without writing a geodatabase."""
    timestamp_test = utc_now()
    assert timestamp_test.endswith("+00:00"), timestamp_test

    valid_4000, reasons_4000 = validate_utm("5136", 4, "7422", "00", 4000, "967")
    assert reasons_4000 == [], reasons_4000
    assert valid_4000["UTM4"] == "00"

    valid_5000, reasons_5000 = validate_utm("5136", 4, "0000", "000", 5000, "967")
    assert reasons_5000 == [], reasons_5000

    _, invalid_scale_reasons = validate_utm("5136", 4, "0000", "00", 4000, "967")
    assert "UTM3_SCALE_MISMATCH" in invalid_scale_reasons

    _, invalid_land_reasons = validate_utm("5136", 4, "7422", "00", 4000, "0")
    assert "INVALID_LAND_NO" in invalid_land_reasons

    assert _minimal_rotation(((2, 2), (1, 1), (3, 3))) == ((1, 1), (3, 3), (2, 2))
    print("Normalization and UTM contract smoke tests passed.")
    item = parse_feature_class_name("A10000000_47")
    assert item["prov_id"] == "10"
    assert item["target_gdb_name"] == "P10.gdb"
    assert "changwat_code" not in item
    assert digit_text("13", 2) == ("13", None)

## 10. Run the export

In [10]:
def validate_parameters():
    if OVERWRITE_EXISTING:
        raise ValueError("OVERWRITE_EXISTING must remain False")
    if PILOT_FIRST_N is not None and (
        isinstance(PILOT_FIRST_N, bool) or not isinstance(PILOT_FIRST_N, int) or PILOT_FIRST_N < 1
    ):
        raise ValueError("PILOT_FIRST_N must be a positive integer or None")
    if SOURCE_NAMES is not None and (
        not isinstance(SOURCE_NAMES, (list, tuple, set))
        or not SOURCE_NAMES or any(not isinstance(name, str) for name in SOURCE_NAMES)
    ):
        raise ValueError("SOURCE_NAMES must be a nonempty list of FC names, or None")
    if not isinstance(REJECT_BATCH_SIZE, int) or REJECT_BATCH_SIZE < 1:
        raise ValueError("REJECT_BATCH_SIZE must be a positive integer")
    if not isinstance(REJECTED_PREVIEW_LIMIT, int) or not 0 <= REJECTED_PREVIEW_LIMIT <= 1_048_575:
        raise ValueError("REJECTED_PREVIEW_LIMIT must be between 0 and 1,048,575")
    if not isinstance(PROGRESS_EVERY, int) or PROGRESS_EVERY < 0:
        raise ValueError("PROGRESS_EVERY must be a nonnegative integer")
    if ALLOW_Z_OR_M:
        raise ValueError("The current canonical output is 2D; ALLOW_Z_OR_M must remain False")


def select_sources(inventory_items):
    selected = inventory_items
    if SOURCE_NAMES is not None:
        requested = {name.upper() for name in SOURCE_NAMES}
        found = {item["source_name"].upper() for item in inventory_items}
        missing = requested - found
        if missing:
            raise ValueError(f"Requested source feature classes not found: {sorted(missing)}")
        selected = [item for item in inventory_items if item["source_name"].upper() in requested]
    return selected[:PILOT_FIRST_N] if PILOT_FIRST_N is not None else selected


def run_export(inventory_items):
    validate_parameters()
    blocking = [item for item in inventory_items if item["preflight_status"] != "OK"]
    if blocking:
        names = ", ".join(item["source_name"] for item in blocking)
        raise RuntimeError(f"Resolve blocking preflight errors before export: {names}")
    if not inventory_items:
        raise RuntimeError("No selected feature classes matched A<BRANCH_CODE>_<ZONE>")
    failures = []
    # State context is validated before an archive directory is created.
    with closing(open_state_database()) as connection:
        for index, item in enumerate(inventory_items, start=1):
            print(f"\n[{index}/{len(inventory_items)}] {item['source_name']}")
            try:
                build_feature_class(connection, item)
            except Exception as error:
                message = "".join(traceback.format_exception(type(error), error, error.__traceback__))
                current = get_feature_state(connection, item["source_name"])
                # Never rewrite incompatible checkpoints. Preserve resumable
                # publication and completed metadata if verification failed.
                if not isinstance(error, CheckpointConflict) and (
                    current is None or current["status"] not in {"BUILT", "PUBLISHED", "COMPLETED"}
                ):
                    upsert_feature_state(connection, item, "FAILED", finished_at=utc_now(), error_message=message)
                record_run_error(connection, item, message)
                failures.append(item["source_name"])
                print(message)
                try_write_excel_report(connection)
                if STOP_ON_ERROR:
                    raise
            else:
                resolve_run_errors(connection, item["source_name"])
                if WRITE_REPORT_AFTER_EACH_FEATURE_CLASS:
                    try_write_excel_report(connection)
        try_write_excel_report(connection)
    if failures:
        raise RuntimeError(
            f"Export attempted all selected sources but {len(failures)} failed: "
            + ", ".join(failures) + ". Inspect ERRORS and RUN_ERRORS."
        )


def main():
    """Inventory first; optionally export. Return (inventory DataFrame, selection)."""
    validate_parameters()
    run_contract_smoke_tests()
    print(f"ArcGIS Pro / ArcPy: {arcpy.GetInstallInfo().get('Version', 'unknown')}")
    print(f"PREPARE_GDB : {PREPARE_GDB}")
    print(f"ARCHIVE_DIR : {ARCHIVE_DIR}")
    print(f"REPORT      : {EXPORT_REPORT}")
    print(f"STATE_DB    : {STATE_DB}")
    print(f"WRITE MODE  : {RUN_EXPORT}")
    print(f"POLICY      : {EXPORT_POLICY_VERSION}")
    # GetCount honors the current processing extent. Clear it for this run so
    # reconciliation covers every source polygon; restore it on context exit.
    with arcpy.EnvManager(extent=None, overwriteOutput=False):
        inventory, ignored = inventory_sources()
        if not inventory:
            raise RuntimeError("No source feature classes matched A<BRANCH_CODE>_<ZONE>")
        inventory_df = pd.DataFrame(inventory)
        display(inventory_df[[
            "source_name", "source_count", "source_wkid", "prov_id", "target_gdb_name",
            "missing_conditional_fields", "preflight_status", "preflight_errors",
        ]])
        selected = select_sources(inventory)
        print(f"Matched feature classes: {len(inventory)}; ignored: {len(ignored)}")
        print(f"Total source rows: {sum(item['source_count'] for item in inventory):,}")
        print(f"Selected {len(selected)} feature classes: {[item['source_name'] for item in selected]}")
        if RUN_EXPORT:
            run_export(selected)
        else:
            print("PRE-FLIGHT ONLY: no archive, state database, or report was changed.")
            print("Inspect preflight_errors. This is schema/count inventory, not a full row-validation run.")
            print("Set RUN_EXPORT=True for a pilot after reviewing parameters.")
        return inventory_df, selected

## Execute

Run this cell after all definitions. Returned inventory and selected sources are available for inspection.

In [11]:
inventory_df, selected_sources = main()

Normalization and UTM contract smoke tests passed.
ArcGIS Pro / ArcPy: 3.7.2
PREPARE_GDB : D:\A02-Projects\Refine\01_acquired\A2022-05.gdb
ARCHIVE_DIR : D:\A02-Projects\Refine\99_temp\_rebuild_service_area_v2
REPORT      : D:\A02-Projects\Refine\05_reports\2022-05\service_area_v2\04_2022-05-Export-service-area-v2.xlsx
STATE_DB    : D:\A02-Projects\Refine\05_reports\2022-05\service_area_v2\04_2022-05-Export-service-area-v2.state.sqlite
WRITE MODE  : True
POLICY      : SERVICE_AREA_ADMIN_V2


,source_name,source_count,source_wkid,prov_id,target_gdb_name,missing_conditional_fields,preflight_status,preflight_errors
0,A10000000_47,119628,24047,10,P10.gdb,,OK,
1,A10010000_47,153112,24047,10,P10.gdb,,OK,
2,A10020000_47,108580,24047,10,P10.gdb,,OK,
3,A10030000_47,103572,24047,10,P10.gdb,,OK,
4,A10040000_47,183657,24047,10,P10.gdb,,OK,
...,...,...,...,...,...,...,...,...
66,A80080000_47,95221,24047,80,P80.gdb,,OK,
67,A80090000_47,71536,24047,80,P80.gdb,,OK,
68,A80100000_47,26594,24047,80,P80.gdb,,OK,
69,A80100100_47,29220,24047,80,P80.gdb,,OK,


Matched feature classes: 71; ignored: 0
Total source rows: 7,224,925
Selected 71 feature classes: ['A10000000_47', 'A10010000_47', 'A10020000_47', 'A10030000_47', 'A10040000_47', 'A10050000_47', 'A10060000_47', 'A10070000_47', 'A10080000_47', 'A10090000_47', 'A10100000_47', 'A10110000_47', 'A10120000_47', 'A10130000_47', 'A10140000_47', 'A10150000_47', 'A10160000_47', 'A20000000_47', 'A20010000_47', 'A20020000_47', 'A20030000_47', 'A20040000_47', 'A20050000_47', 'A30000000_47', 'A30000000_48', 'A30010000_48', 'A30020000_48', 'A30030000_48', 'A30040000_47', 'A30040000_48', 'A30050000_48', 'A30060000_48', 'A30070000_48', 'A30080000_47', 'A30090000_47', 'A30100000_47', 'A30110000_48', 'A30120000_48', 'A30130000_47', 'A30140000_47', 'A30140000_48', 'A30150000_48', 'A30160000_48', 'A30170000_47', 'A30170000_48', 'A50000000_47', 'A50010000_47', 'A50010100_47', 'A50010200_47', 'A50020000_47', 'A50030000_47', 'A50030100_47', 'A50040000_47', 'A50050000_47', 'A50060000_47', 'A50070000_47', 'A500